# 02 — Retrieve documents from PostgreSQL
This lab connects directly to the live workshop database. `sources` stores document metadata/Blob links; `chunks` stores searchable text and embeddings.

In [ ]:
from course_knowledge.database import connect, vector_literal
from course_knowledge.settings import database_settings, azure_openai_settings
from course_knowledge.embeddings import AzureOpenAIEmbedder
db = connect(database_settings())
with db.cursor() as cur:
    cur.execute('SELECT canonical_filename, source_type, storage_uri FROM sources ORDER BY canonical_filename LIMIT 10')
    for row in cur.fetchall(): print(row)

In [ ]:
question = 'How does Model Context Protocol work?'
vector = AzureOpenAIEmbedder.from_settings(azure_openai_settings()).embed([question])[0]
with db.cursor() as cur:
    cur.execute('SELECT id::text, left(text_content, 180) AS excerpt, embedding <=> %s::vector AS distance FROM chunks WHERE embedding IS NOT NULL ORDER BY embedding <=> %s::vector LIMIT 5', (vector_literal(vector), vector_literal(vector)))
    for row in cur.fetchall(): print(row)

## Exercise
Change the question. Explain why a lower cosine distance is more relevant.